In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import mkdir, show
import json
from PIL import Image
from pycocotools import mask as mask_utils
from tqdm import tqdm
from mtrain.disk import DiskBooleanMask, DiskImage
import itertools

In [ ]:
ALL_DELHI_IMAGES = Path("/Users/hariomnarang/Desktop/personal/roads/mapillary_downloader/data/delhi/images")
DEST_DIR = Path("../../datasets/inference/delhi_sample_5000")

all_images = list(ALL_DELHI_IMAGES.glob("*.jpg"))
samples = random.sample(all_images, 5000)

In [ ]:
SAMPLE_DS = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test")

In [ ]:
dirs = list(SAMPLE_DS.glob("*"))
print("length all dirs", len(dirs))
dirs = random.sample(dirs, 10)

In [ ]:
image, mask = DiskImage.load(dirs[0] / "image.jpg"), DiskBooleanMask.load(dirs[0] / "m2.png")

In [ ]:
show([image, mask])

In [ ]:
# plt.imshow(plt.imread("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/3200856443484789"))

In [ ]:
import shutil
for p in samples:
    dest = mkdir(DEST_DIR / p.stem)
    shutil.copy(p, dest / "image.jpg")
    

In [ ]:
DS = Path("../../datasets/")
BASE = DS / "test-samples"
NEG_MASKING_V1 = BASE / "neg-masking" / "V1"
TRASH = NEG_MASKING_V1 / "trash"
SAMPLES_MAPILLARY = NEG_MASKING_V1 / "samples_mapillary"
MODEL_DIR = DS / "models" / "trash_classification"

CLIP_FILE_NAMES = [
    "clip_flowers.txt",
    # "clip_fallen_leaves.txt",
    # "clip_bottles.txt",
    # "clip_litter.txt",
    # "clip_plastic.txt",
    # "clip_tobacco_packs.txt",
    # "clip_delhi_litter.txt",
]
CLIP_FILES = [TRASH / c for c in CLIP_FILE_NAMES]
print("all clip files exist:", all([f.exists() for f in CLIP_FILES]))

TRASH_DATA_DIR = TRASH / "data"

SAMPLES_MAPILLARY.exists(), TRASH.exists()

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_2 import LabelWidget
from mtrain.neg_mask.crops import get_region_crops, Bbox

In [ ]:
def read_clip_file(path) -> list[tuple[str, Path]]:
    with open(path) as f:
        lines = f.readlines()
    imgs = [Path(line.split("\t")[1].strip()) for line in lines]
    dirs = [(path.stem, img.parent) for img in imgs]
    return dirs


def get_all_from_dir(path) -> list[tuple[str, Path]]:
    return [(path.stem, d) for d in path.glob("*")]


def get_dir_and_cat_iter():
    all_cat_dirs = [read_clip_file(f) for f in CLIP_FILES]
    # all_cat_dirs.append(get_all_from_dir(TRASH / "personal"))
    return itertools.chain.from_iterable(itertools.zip_longest(*all_cat_dirs))


def get_dir_for_widget():
    return (d for (_, d) in get_dir_and_cat_iter())

In [ ]:
images = [p[1] for p in read_clip_file(TRASH / "clip_delhi_litter.txt")]

In [ ]:
idx = 0

In [ ]:
litters = [
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/26210846388517270/image.jpg"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/933090527327250/image.jpg"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/904411971827702/image.jpg"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/409495874297166/image.jpg"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/471611377923092/image.jpg"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/1132072153938604/image.jpg"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/318666179638135/image.jpg"
    ),
]

In [ ]:
Image.open(litters[0])

# Classfication dataset

In [ ]:
from mtrain.neg_mask.openai_clip import get_images_from_clip_file
CLIP_NAME = "delhi_litter"
CLIP_FILE = f"/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/clip_{CLIP_NAME}.txt"
CLS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/crop_level")


TOTAL_SAMPLES = 50
DEST_DIR = Path(
    f"/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/{CLIP_NAME}"
)

def get_images_not_in_train_set():
    images = list(get_images_from_clip_file(CLIP_FILE))
    all_dirs = set()
    for label in ["other", "trash"]:
        sample_names_without_crop_idx = set((p.name.split("_")[0] for p in (CLS_DIR / label).glob("*") if p.is_dir()))
        for s in sample_names_without_crop_idx:
            all_dirs.add(s)
    return [img for img in images if img.stem not in all_dirs]

In [ ]:
paths = get_images_not_in_train_set()

In [ ]:
DEST_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/full_image_runs/delhi_litter"
)

In [ ]:
res = []
for d in DEST_DIR.glob("*"):
    image = DiskImage.load(d / "image.jpg")
    mask = DiskBooleanMask.load(d / "mask.png")
    trash_probs = np.load(d / "trash_probs.npy")
    other_probs = np.load(d / "other_probs.npy")
    res.append((image, mask, trash_probs, other_probs))

In [ ]:
plt.imshow(plt.imread("/Users/hariomnarang/Downloads/mask.png"))

In [ ]:
show(res[0], (30,30), ncols=4, axis="off")

In [ ]:
# from mtrain.neg_mask.ipywidgets.widget_3 import EvalWidget
from mtrain.neg_mask.model.learner import load_our_learner, dummy_dls
from mtrain.neg_mask.model.crop_level_dataset import CropLevelDataset2Chan
from mtrain.neg_mask.openai_clip import interleaved_data_from_multiple_clip_files
from mtrain.neg_mask.model.predict.predict_8ch import predict_and_reconstruct_mask
from mtrain.smallnet.unet.predict.strided.single import strided_predict_unet_only_mask
from fastai.vision.all import (
    mobilenet_v3_large,
    resnet18,
    vision_learner,
    accuracy,
    F1Score,
    CrossEntropyLossFlat,
    ProgressCallback,
    DataLoaders,
    default_device,
)

LABELS = ["other", "trash"]
DataLoaders.from_dsets(
    CropLevelDataset2Chan([], LABELS, True, medium_pad=220),
    CropLevelDataset2Chan([], LABELS, False, medium_pad=220),
)

learner = vision_learner(
    dummy_dls(LABELS),
    resnet18,
    n_in=8,
    metrics=[accuracy, F1Score(average="macro")],
    loss_func=CrossEntropyLossFlat(),
    n_out=len(LABELS),
    normalize=False,
)
learner = learner.remove_cb(ProgressCallback)

learner = learner.load(
    # "resnet18-size_130-chan_8-with_augs-iter_40.pth"
    "/Users/hariomnarang/Desktop/personal/roads/datasets/models/trash_classification/resnet18-size_130-chan_8-with_augs-iter_40"
)

# learn = load_our_learner(dls, mobilenet_v3_large, None, LABELS)
# state_dict = torch.load('/Users/hariomnarang/Desktop/personal/roads/datasets/models/taco_pretrained_mask_classifier/mobilenet_v3_large_130x130_iter-20-pure-torch.pth')
# keys_to_remove = [k for k in state_dict.keys() if '1.8' in k]
# print("Removing:", keys_to_remove)
# for k in keys_to_remove:
#     del state_dict[k]

# learn.model.load_state_dict(state_dict, strict=False)

In [ ]:
# from mtrain.neg_mask.ipywidgets.bbox_processing import get_region_crops

# def predict_and_return_probs(image, mask, learner, crop_pad):
#     """Run inference on all bounding boxes and create probability masks."""
#     bboxes = list(get_region_crops(image, mask))
#     mask = mask.astype(bool)
#     if not bboxes or learner is None:
#         h, w = mask.shape
#         trash_prob_mask = np.zeros((h, w), dtype=np.float32)
#         other_prob_mask = np.zeros((h, w), dtype=np.float32)
#         return trash_prob_mask, other_prob_mask

#     from mtrain.neg_mask.model.predict.predict_8ch import run_inference

#     # Prepare crop data for batch inference
#     crop_data_list = [(image, mask, bbox) for bbox in bboxes]

#     # Run batch inference
#     all_probs = run_inference(learner, crop_data_list, crop_pad)  # Shape: [N, C]
#     return reconstruct_probability_masks(mask, all_probs.numpy(), bboxes, 0, 1)


# def reconstruct_probability_masks(mask, all_probs, bboxes, label_other, label_trash):
#     """Map bbox predictions back to full image coordinates."""
#     h, w = mask.shape
#     trash_prob_mask = np.zeros((h, w), dtype=np.float32)
#     other_prob_mask = np.zeros((h, w), dtype=np.float32)

#     for i, bbox in enumerate(bboxes):
#         # Get probabilities for this bbox
#         other_prob = all_probs[i, label_other].item()
#         trash_prob = all_probs[i, label_trash].item()

#         # Apply to bbox region in full image
#         bbox_mask = mask[bbox.y : bbox.y2, bbox.x : bbox.x2]
#         other_prob_mask[bbox.y : bbox.y2, bbox.x : bbox.x2][bbox_mask] = (
#             other_prob
#         )
#         trash_prob_mask[bbox.y : bbox.y2, bbox.x : bbox.x2][bbox_mask] = (
#             trash_prob
#         )

#     return trash_prob_mask, other_prob_mask

# def get_thrash_mask_above_threshold_and_other(
#     mask, trash_prob_mask, other_prob_mask, trash_thres
# ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
#     """Create overlay showing the predicted class (highest probability) for each pixel."""
#     h, w = mask.shape

#     # Find pixels where we have predictions
#     has_prediction = (trash_prob_mask > 0) | (other_prob_mask > 0)
#     trash_above_other = trash_prob_mask >= other_prob_mask
#     trash_above_thres = trash_prob_mask > trash_thres
#     return trash_above_other, trash_above_thres, has_prediction


In [ ]:
%matplotlib inline
from mtrain.neg_mask.model.predict.full_image_8chan import predict_and_return_probs, get_thrash_mask_above_threshold_and_other
from mtrain.utils import show

flower_image = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1253130721794975/image.jpg")
image, mask = DiskImage.load(flower_image),DiskBooleanMask.load(flower_image.parent / "mask.png")
trash_probs, other_probs = predict_and_return_probs(
    image, mask, 
    learner,
    220,
)
trash_above_other, trash_above_thres, has_prediction = get_thrash_mask_above_threshold_and_other(mask, trash_probs, other_probs, 0.7)

show([
    mask, trash_probs, other_probs,
    trash_above_other, trash_above_thres, image,
], (30,30), 3, "off")

In [ ]:
%matplotlib inline
from mtrain.neg_mask.model.predict.full_image_8chan import predict_and_return_probs, get_thrash_mask_above_threshold_and_other
from mtrain.utils import show

image, mask = DiskImage.load(),DiskBooleanMask.load(litters[0].parent / "mask.png")
trash_probs, other_probs = predict_and_return_probs(
    image, mask, 
    learner,
    220,
)
trash_above_other, trash_above_thres, has_prediction = get_thrash_mask_above_threshold_and_other(mask, trash_probs, other_probs, 0.7)

show([
    mask, trash_probs, other_probs,
    trash_above_other, trash_above_thres, has_prediction,
], (30,30), 3, "off")

In [ ]:
%matplotlib inline
from mtrain.utils import show

show([
    mask, trash_probs, other_probs,
    trash_above_other, trash_above_thres, has_prediction,
], (30,30), 3, "off")

In [ ]:
# from mtrain.neg_mask.model.predict.predict_8ch import (
#     run_inference,
# )
# from mtrain.neg_mask.crops import get_crops_for_image, get_region_crops

# def reconstruct_probability_masks(
#     image, mask, all_probs, bboxes, label_other=0, label_trash=1
# ):
#     """Map bbox predictions back to full image coordinates."""
#     h, w = mask.shape
#     trash_prob_mask = np.zeros((h, w), dtype=np.float32)
#     other_prob_mask = np.zeros((h, w), dtype=np.float32)

#     for i, bbox in enumerate(bboxes):
#         # Get probabilities for this bbox
#         other_prob = all_probs[i, label_other].item()
#         trash_prob = all_probs[i, label_trash].item()

#         # Apply to bbox region in full image
#         print(bbox, other_prob_mask.shape, trash_prob_mask.shape)
#         bbox_mask = mask[bbox.y : bbox.y2, bbox.x : bbox.x2]
#         other_prob_mask[bbox.y : bbox.y2, bbox.x : bbox.x2][bbox_mask] = other_prob
#         trash_prob_mask[bbox.y : bbox.y2, bbox.x : bbox.x2][bbox_mask] = trash_prob
#     return trash_prob_mask, other_prob_mask

# def predict_and_return_probs(
#     learn,
#     image: np.ndarray,
#     mask: np.ndarray,
#     trash_thres,
#     bbox_pad=20,
#     crop_pad=220,
#     label_other=0,
#     label_trash=0,
# ):
#     bboxes = list(get_region_crops(image, mask))
#     if not bboxes:
#         return mask.copy().astype(np.float32)
    
#     crop_data_list = [(image, mask, bbox) for bbox in bboxes]
#     all_probs = run_inference(learn, crop_data_list, crop_pad)  # Shape: [N, C]


#     print(mask.shape, all_probs.shape, len(bboxes))
#     trash_prob_mask, other_prob_mask = reconstruct_probability_masks(
#         image, mask, all_probs, bboxes, label_other, label_trash
#     )
#     return trash_prob_mask, other_prob_mask
#     # trash_above_other, trash_above_thres, has_prediction = (
#     #     get_thrash_mask_above_threshold_and_other(
#     #         mask, trash_prob_mask, other_prob_mask, trash_thres
#     #     )
#     # )
#     # return trash_above_other, trash_above_thres, has_prediction

In [ ]:
# img, mask = DiskImage.load(litters[0]),DiskBooleanMask.load(litters[0].parent / "mask.png")
# print(img.shape, mask.shape)
# trash_prob_mask, other_prob_mask = predict_and_return_probs(
#     learner,
#     img, mask, 
    
#     0.8,
# )

In [ ]:
# it = iter(get_all_from_dir(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/personal")))
# it = (d for (n,d) in it)

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_6 import MassAnnotationWidget
from mtrain.neg_mask.ipywidgets.widget_2 import LabelWidget

# Create widget


ROCKS = NEG_MASKING_V1 / "rocks"
# classification folder is here
widget = LabelWidget(ROCKS / "classification-test", learner=learner)
# widget = MassAnnotationWidget(ROCKS / "classification", learner, 220)


def is_dir_not_done(direc: Path):
    return not widget.is_done(direc.name)


# widget = LabelWidget(ROCKS / "classification", crop_pad=220, learner=learner)
# widget = LabelWidget(ROCKS / "classification", crop_pad=220)


it = get_all_from_dir(ROCKS / "classification" / "flowers")
it = (d[1] for d in it)
# it = get_dir_for_widget()
# it = filter(is_dir_not_done, it)


# dir_it = filter(is_dir_not_done, iter(get_dir_for_widget()))

# run the cell below for vscode color theme support in ipywidgets

# Source - https://stackoverflow.com/a/77028015
# Posted by Yingding Wang, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-02, License - CC BY-SA 4.0

In [ ]:
# %matplotlib ipympl

In [ ]:
def _get_next_dir_assets(d):
    img, mask = DiskImage.load(d / "image.jpg"), DiskBooleanMask.load(d / "m2.png")
    bboxes = list(get_region_crops(img, mask))
    return img, mask, bboxes


d = next(it)
img, mask, bboxes = _get_next_dir_assets(d)
while len(bboxes) == 0:
    print("SKIP:", d.name)
    d = next(it)
# d = litters[0].parent
# img, mask, bboxes = _get_next_dir_assets(d)


widget.ui(d, bboxes, img, mask)


In [ ]:
from mtrain.neg_mask.ipywidgets.widget_6 import read_saved_dataset
from pathlib import Path

test_it = read_saved_dataset(ROCKS / "test-classification")

# Model performance analysis

In [ ]:
from mtrain.neg_mask.ipywidgets.widget_3 import EvalWidget
from mtrain.neg_mask.model.learner import load_our_learner, dummy_dls
from mtrain.neg_mask.openai_clip import interleaved_data_from_multiple_clip_files
from fastai.vision.all import mobilenet_v3_large

model = (MODEL_DIR / "mobilenet_large_mask_thres_6_25_epochs").resolve()
learner = load_our_learner(dummy_dls(), mobilenet_v3_large, None, model)

In [ ]:
import itertools


def widget_iter(files):
    for name, image_dir in interleaved_data_from_multiple_clip_files(files):
        yield (image_dir / "image.jpg").resolve(), (image_dir / "m2.png").resolve()


iterator = widget_iter(CLIP_FILES)

In [ ]:
widget = EvalWidget(learner, iterator)

In [ ]:
# generate the masks first
from fastai.vision.all import load_learner

learner100 = load_learner(
    "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/enguled-bbox-levels-crops-v3/log/export_iter_14.pkl"
)
SIZE = 100

In [ ]:
from mtrain.smallnet.unet.predict.strided import single
from mtrain.disk import DiskImage, DiskBooleanMask
from tqdm import tqdm

img_dirs = list((TRASH / "delhi_litter").glob("*"))
for d in tqdm(img_dirs):
    img = DiskImage.load(d / "image.jpg")
    mask = single.strided_predict_unet_only_mask(img, 100, learner100, [50])
    DiskBooleanMask.save(mask, "m2.png")

In [ ]:
from mtrain.neg_mask.openai_clip import (
    get_images_from_clip_file,
    get_image_dirs_from_clip_file,
)

for _, d in tqdm(get_image_dirs_from_clip_file(TRASH / "clip_litter.txt")[:100]):
    img = DiskImage.load(d / "image.jpg")
    mask = single.strided_predict_unet_only_mask(img, 100, learner100, [50])
    DiskBooleanMask.save(mask, "m2.png")